# 01a · Tokens: a first look

The most elementary operation of the NER series: **give a tokenizer some
text, look at the tokens it returns.** One routine, and a menu of
tokenizers to plug into it.

The detailed comparison (algorithms, tables, corpus measures, entity
boundaries) is in `01b-tokenization.ipynb`.

## Preparation

Every tokenizer below offers the same method, `tokenizer.tokenize(text)`,
which returns a list of strings (the tokens, without special tokens such as
`[CLS]` and `[SEP]`). The Hugging Face tokenizers have it already; for spaCy
we write a three-line adapter.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import spacy
import transformers
from transformers import AutoTokenizer

transformers.logging.set_verbosity_error()


class SpacyTokenizer:
    # gives a spaCy tokenizer the same .tokenize(text) method as the Hugging Face ones
    def __init__(self, model):
        self.nlp = spacy.load(model) if model != "blank" else spacy.blank("en")
    def tokenize(self, text):
        return [t.text for t in self.nlp.tokenizer(text)]


TOKENIZERS = {
    # name                          tokenizer                                         approach
    "Rule-based (scispaCy)":        SpacyTokenizer("en_core_sci_sm"),                 # rules for biomedical text
    "Rule-based (spaCy)":           SpacyTokenizer("blank"),                          # spaCy's default English rules
    "WordPiece (BERT, uncased)":    AutoTokenizer.from_pretrained("d4data/biomedical-ner-all"),  # used in ner_pipeline
    "WordPiece (BiomedBERT)":       AutoTokenizer.from_pretrained(
                                        "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"),
    "BPE (RoBERTa)":                AutoTokenizer.from_pretrained("roberta-base"),
    "Unigram (ALBERT)":             AutoTokenizer.from_pretrained("albert-base-v2"),
    "Bytes (ByT5)":                 AutoTokenizer.from_pretrained("google/byt5-small"),
}

# The walkthrough sentence of ner_pipeline.ipynb (case PMC5137649_01 of the MultiCaRe sample).
sentence = ("A 44-year-old woman presented with a 3-day history of right flank and lower "
            "quadrant abdominal pain associated with nausea and constipation.")
WORDS = ["constipation", "hypokalemia", "dyspnea", "echocardiography", "44-year-old"]

print("available tokenizers:")
for name in TOKENIZERS:
    print("  ", name)

available tokenizers:
   Rule-based (scispaCy)
   Rule-based (spaCy)
   WordPiece (BERT, uncased)
   WordPiece (BiomedBERT)
   BPE (RoBERTa)
   Unigram (ALBERT)
   Bytes (ByT5)


## The routine

Pick a tokenizer by name, then tokenize the sentence and a few clinical
words. **Change the name in the first line** (copy one from the list above)
and run the cell again.

In [2]:
tokenizer = TOKENIZERS["WordPiece (BERT, uncased)"]     # ← change me

pieces = tokenizer.tokenize(sentence)
print("tokens in the sentence:", len(pieces), "\n")
for w in WORDS:
    print(f"{w:18s} → {tokenizer.tokenize(w)}")

tokens in the sentence: 31 

constipation       → ['con', '##sti', '##pati', '##on']
hypokalemia        → ['h', '##yp', '##oka', '##lem', '##ia']
dyspnea            → ['d', '##ys', '##p', '##nea']
echocardiography   → ['echo', '##card', '##iography']
44-year-old        → ['44', '-', 'year', '-', 'old']


## The same routine, for every tokenizer

In [3]:
for name, tokenizer in TOKENIZERS.items():
    print(f"══ {name}")
    pieces = tokenizer.tokenize(sentence)
    print("tokens in the sentence:", len(pieces), "\n")
    for w in WORDS:
        print(f"{w:18s} → {tokenizer.tokenize(w)}")
    print()

══ Rule-based (scispaCy)
tokens in the sentence: 22 

constipation       → ['constipation']
hypokalemia        → ['hypokalemia']
dyspnea            → ['dyspnea']
echocardiography   → ['echocardiography']
44-year-old        → ['44-year-old']

══ Rule-based (spaCy)
tokens in the sentence: 28 

constipation       → ['constipation']
hypokalemia        → ['hypokalemia']
dyspnea            → ['dyspnea']
echocardiography   → ['echocardiography']
44-year-old        → ['44', '-', 'year', '-', 'old']

══ WordPiece (BERT, uncased)
tokens in the sentence: 31 

constipation       → ['con', '##sti', '##pati', '##on']
hypokalemia        → ['h', '##yp', '##oka', '##lem', '##ia']
dyspnea            → ['d', '##ys', '##p', '##nea']
echocardiography   → ['echo', '##card', '##iography']
44-year-old        → ['44', '-', 'year', '-', 'old']

══ WordPiece (BiomedBERT)
tokens in the sentence: 28 

constipation       → ['constipation']
hypokalemia        → ['hypo', '##ka', '##le', '##mia']
dyspnea            → 

## What to notice

* **Rule-based** tokenizers never split a plain word; they only cut at
  spaces and punctuation, by rules — and scispaCy's rules keep
  `44-year-old` in one piece.
* **WordPiece** marks a piece that continues a word with `##`. Compare the
  general vocabulary (BERT) with the biomedical one (BiomedBERT) on the same
  words.
* **Unigram (ALBERT)** marks the *start* of a word instead: `▁` stands for
  the space before it. **BPE (RoBERTa)** does the same with `Ġ`, but only
  when a space really precedes the word — a word tokenized alone, as here,
  has none. Try `tokenizer.tokenize(" " + w)`.
* **Bytes** have no vocabulary to learn: one token per byte of text.

Why they behave like this: `01b-tokenization.ipynb`.